# BLS Preprocessing

In [5]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path


In [20]:
warnings.filterwarnings("ignore")

RAW_DIR = Path("../../data/raw/bls")
OUT_DIR = Path("../../data/processed/bls")

FILES = {
    2019: RAW_DIR / "oesm19nat.xlsx",
    2022: RAW_DIR / "oesm22nat.xlsx",
    2024: RAW_DIR / "oesm24nat.xlsx",
}


## Aggregation

### 1. Load

In [21]:
frames = []
for year, path in FILES.items():
    df = pd.read_excel(path, sheet_name=0)
    df.columns = df.columns.str.upper().str.strip()
    df["YEAR"] = year
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)


In [22]:
detail = raw[raw["O_GROUP"] == "detailed"].copy()

WAGE_COLS = [
    "H_MEAN", "A_MEAN",
    "H_PCT10", "H_PCT25", "H_MEDIAN", "H_PCT75", "H_PCT90",
    "A_PCT10", "A_PCT25", "A_MEDIAN", "A_PCT75", "A_PCT90",
]

for col in WAGE_COLS:
    detail[col] = pd.to_numeric(detail[col], errors="coerce")
detail["TOT_EMP"] = pd.to_numeric(detail["TOT_EMP"], errors="coerce")

detail["MAJOR_SOC"] = detail["OCC_CODE"].str[:2]
major_titles = (
    raw[raw["O_GROUP"] == "major"]
    .drop_duplicates("OCC_CODE")
    .set_index("OCC_CODE")["OCC_TITLE"]
    .to_dict()
)
detail["MAJOR_TITLE"] = detail["MAJOR_SOC"].map(
    lambda code: major_titles.get(f"{code}-0000", "Unknown")
)

detail.head()


,AREA,AREA_TITLE,AREA_TYPE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,O_GROUP,...,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,YEAR,PRIM_STATE,PCT_RPT,MAJOR_SOC,MAJOR_TITLE
4,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,detailed,...,184460.0,NaN,NaN,NaN,NaN,2019,NaN,NaN,11,Management Occupations
6,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,detailed,...,100780.0,157430.0,NaN,NaN,NaN,2019,NaN,NaN,11,Management Occupations
8,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-1031,Legislators,detailed,...,29270.0,75520.0,100470.0,True,NaN,2019,NaN,NaN,11,Management Occupations
11,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-2011,Advertising and Promotions Managers,detailed,...,125510.0,175940.0,NaN,NaN,NaN,2019,NaN,NaN,11,Management Occupations
13,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,detailed,...,136850.0,185320.0,NaN,NaN,NaN,2019,NaN,NaN,11,Management Occupations


### 2. Statistical Aggregation

In [23]:
def safe_mode(s):
    m = s.dropna().mode()
    return m.iloc[0] if len(m) else np.nan

stat_agg = (
    detail.groupby(["MAJOR_SOC", "MAJOR_TITLE", "YEAR"])
    .agg(
        emp_sum=("TOT_EMP", "sum"),
        wage_mean=("A_MEAN", "mean"),
        wage_median=("A_MEAN", "median"),
        wage_mode=("A_MEAN", safe_mode),
        wage_std=("A_MEAN", "std"),
        wage_var=("A_MEAN", "var"),
        wage_q1=("A_MEAN", lambda x: np.nanpercentile(x, 25)),
        wage_q3=("A_MEAN", lambda x: np.nanpercentile(x, 75)),
    )
    .reset_index()
    .round(0)
)

stat_agg.head()

,MAJOR_SOC,MAJOR_TITLE,YEAR,emp_sum,wage_mean,wage_median,wage_mode,wage_std,wage_var,wage_q1,wage_q3
0,11,Management Occupations,2019,8054130,110622.0,113755.0,49440.0,33885.0,1.148185e+09,83072.0,133815.0
1,11,Management Occupations,2022,9860710,118032.0,115410.0,57610.0,39372.0,1.550135e+09,84335.0,145098.0
2,11,Management Occupations,2024,10966830,126781.0,125240.0,62640.0,42248.0,1.784904e+09,94630.0,154830.0
3,13,Business and Financial Operations Occupations,2019,8183760,74340.0,71570.0,49550.0,15099.0,2.279769e+08,65640.0,80220.0
4,13,Business and Financial Operations Occupations,2022,9677710,83041.0,80840.0,51650.0,19416.0,3.769677e+08,72705.0,88805.0


### 3. Temporal Aggregation

In [29]:
temporal_agg = (
    stat_agg.groupby(["MAJOR_SOC", "MAJOR_TITLE"])
    .agg(
        emp_mean=("emp_sum", "mean"),
        wage_mean=("wage_mean", "mean"),
        wage_std=("wage_mean", "std"),
    )
    .reset_index()
    .round(0)
)

temporal_agg.head()

,MAJOR_SOC,MAJOR_TITLE,emp_mean,wage_mean,wage_std
0,11,Management Occupations,9627223.0,118478.0,8089.0
1,13,Business and Financial Operations Occupations,9404300.0,82671.0,8152.0
2,15,Computer and Mathematical Occupations,4916557.0,106098.0,8520.0
3,17,Architecture and Engineering Occupations,2547017.0,91254.0,6554.0
4,19,"Life, Physical, and Social Science Occupations",1350017.0,86447.0,5664.0


### 4. Save

In [30]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
AGG_DIR = OUT_DIR / "agg"
AGG_DIR.mkdir(parents=True, exist_ok=True)

detail.to_csv(AGG_DIR / "bls_detail_clean.csv", index=False)
stat_agg.to_csv(AGG_DIR / "bls_stat_agg.csv", index=False)
temporal_agg.to_csv(AGG_DIR / "bls_temporal_agg.csv", index=False)

for f in sorted(AGG_DIR.glob("bls_*.csv")):
    print(f"  {f.name:30s} {f.stat().st_size / 1024:.1f} KB")

  bls_detail_clean.csv           607.9 KB
  bls_stat_agg.csv               7.7 KB
  bls_temporal_agg.csv           1.5 KB


## Sampling

### 1. Random Sample (10% Rule)

In [32]:
N = len(detail)
n = int(N * 0.10)

sample = detail.sample(n=n, random_state=42)

print(f"Population: {N}  |  Sample: {n}  |  Ratio: {n/N:.0%}")
sample.head()

Population: 2450  |  Sample: 245  |  Ratio: 10%


,AREA,AREA_TITLE,AREA_TYPE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,O_GROUP,...,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,YEAR,PRIM_STATE,PCT_RPT,MAJOR_SOC,MAJOR_TITLE
2175,99,U.S.,1,0,Cross-industry,cross-industry,1235,43-3011,Bill and Account Collectors,detailed,...,39470.0,48290.0,59610.0,NaN,NaN,2022,US,NaN,43,Office and Administrative Support Occupations
1674,99,U.S.,1,0,Cross-industry,cross-industry,1235,23-2011,Paralegals and Legal Assistants,detailed,...,59200.0,75560.0,94960.0,NaN,NaN,2022,US,NaN,23,Legal Occupations
3992,99,U.S.,1,0,Cross-industry,cross-industry,1235,51-8031,Water and Wastewater Treatment Plant and Syste...,detailed,...,58260.0,71280.0,86160.0,NaN,NaN,2024,US,NaN,51,Production Occupations
1089,99,U.S.,1,0,Cross-industry,cross-industry,1235,51-2041,Structural Metal Fabricators and Fitters,detailed,...,40390.0,50130.0,61500.0,NaN,NaN,2019,NaN,NaN,51,Production Occupations
1035,99,U.S.,1,0,Cross-industry,cross-industry,1235,49-3052,Motorcycle Mechanics,detailed,...,37600.0,48530.0,60060.0,NaN,NaN,2019,NaN,NaN,49,"Installation, Maintenance, and Repair Occupations"


### 2. CLT — Distribution of Sample Means

In [35]:
wages = detail["A_MEAN"].dropna()

n_samples = 1000
sample_means = [wages.sample(n=n, replace=False).mean() for _ in range(n_samples)]
sample_means = pd.Series(sample_means)

pop_mu = wages.mean()
mu_xbar = sample_means.mean()
se = wages.std() / np.sqrt(n)

print(f"Population μ:            {pop_mu:,.0f}")
print(f"Mean of sample means μ_x̄: {mu_xbar:,.0f}")
print(f"Std of sample means:     {sample_means.std():,.0f}")
print(f"σ/√n:                    {se:,.0f}")

Population μ:            71,020
Mean of sample means μ_x̄: 71,132
Std of sample means:     2,725
σ/√n:                    2,889


### 3. Save

In [34]:
SAMP_DIR = OUT_DIR / "sampling"
SAMP_DIR.mkdir(parents=True, exist_ok=True)

sample.to_csv(SAMP_DIR / "bls_random_sample.csv", index=False)

for f in sorted(SAMP_DIR.glob("bls_*.csv")):
    print(f"  {f.name:30s} {f.stat().st_size / 1024:.1f} KB")

  bls_random_sample.csv          61.4 KB
